## Импорт зависимостей

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

In [2]:
pd.set_option('display.max_columns', 50)

## Загрузка данных и разделение набора данных

In [3]:
from sklearn.model_selection import train_test_split

DATA_PATH = Path('../data')
TRAIN_PATH = DATA_PATH / 'train.csv'
TEST_PATH = DATA_PATH / 'test.csv'

TARGET = 'addicted_label'

TRAIN_DATA = pd.read_csv(TRAIN_PATH, index_col='id')
TEST_DATA = pd.read_csv(TEST_PATH, index_col='id')

X = TRAIN_DATA.drop(columns=TARGET)
y = TRAIN_DATA[TARGET]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)
X_test = TEST_DATA


In [4]:
X.shape

(691369, 12)

In [5]:
X.duplicated().sum()

np.int64(0)

## Понимание природы данных, их представления в датасете и количестве пропусков, дубликатов

Каждая строка данных представляет собой каждого отдельно взятого человека(всего их 691369). Нужно предсказать, зависим человек от телефона или нет

Строки(их значение в датасете и в реале):
- id - идентификатор каждого из участников - (нет пропусков)
- age - возраст - непрерывный числовой признак - (есть пропуски)
- daily_screen_time_hours - часы за экраном каждый день - непрерывный числовой признак - (есть пропуски)
- social_media_hours - часы в соцсетях - непрерывный числовой признак - (есть пропуски)
- gaming_hours - часы в играх - непрерывный числовой признак - (есть пропуски)
- work_study_hours - часы за работой/учебой - непрерывный числовой признак - (есть пропуски)
- sleep_hours - время сна - непрерывный числовой признак - (есть пропуски)
- notifications_per_day - количество уведомлений - счётный числовой признак - (есть пропуски)
- app_opens_per_day - количество приложений открытых за день - счетный числовой признак - (есть пропуски)
- weekend_screen_time - время за экраном в выходной - непрерывный числовой признак - (есть пропуски)
- gender - пол - номинальный категориальный признак - (есть пропуски) - мужчина, женщина, другой
- stress_level - уровень стресса - номинальный категориальный признак - (есть пропуски) - низкий, средний, высокий
- academic_work_impact - негативное влияние телефона на работу/учебу - номинальный категориальный признак  - есть пропуски - да, нет


### Неправильно
Идеи для преобразования данных:
- Признак - соотношение рабочих часов за экраном и количество часов в нерабочее время


### Правильно
Поскольку я буду делать упор на модели с градиентным бустингом, то в признаках-соотношениях других признаков смысла мало, тк такие модели сами находят линейные и нелинейные зависимости - нужно делать что-то другое:
- Признак - значение оставшегося времени, которое указано, как экранное, но не подходит под одну из выбранных категорий(соцсети, игры, работа)


In [6]:
X_train.head(10)

,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact
id,,,,,,,,,,,,
285927,24.0,7.25,2.95,1.46,2.06,8.14,65.0,155.0,11.54,Male,Medium,Yes
285459,27.0,10.17,3.20,2.42,3.27,8.12,234.0,114.0,11.54,Female,Low,Yes
362545,27.0,2.90,0.78,0.22,1.57,6.60,142.0,39.0,5.67,NaN,NaN,NaN
268214,26.0,10.30,NaN,2.45,1.35,7.74,135.0,35.0,12.68,Other,Medium,Yes
71572,30.0,7.25,3.39,NaN,1.95,6.36,NaN,NaN,NaN,Male,High,Yes
15457,22.0,NaN,2.60,0.22,1.27,5.15,201.0,94.0,9.50,Female,Medium,Yes
220555,27.0,5.77,2.45,1.81,0.92,5.79,24.0,132.0,8.53,Male,Medium,Yes
680126,30.0,10.29,2.80,2.42,4.46,8.29,236.0,125.0,12.37,Male,Low,No
397807,25.0,3.41,0.83,0.70,1.65,5.89,215.0,78.0,5.69,Female,Medium,Yes


In [7]:
isnas = X_train['academic_work_impact'].isna().value_counts()
dups = X_train['academic_work_impact'].duplicated().sum()
counts_cats = X_train['academic_work_impact'].value_counts()
print(f"""-----
isNA:
-----
{isnas}
-----------
Duplicates:
-----------
{dups} / {X_train.shape[0]}
------------------------------
Value counts(for categorical):
------------------------------
{counts_cats}""")

-----
isNA:
-----
academic_work_impact
False    582292
True      39940
Name: count, dtype: int64
-----------
Duplicates:
-----------
622229 / 622232
------------------------------
Value counts(for categorical):
------------------------------
academic_work_impact
Yes    297553
No     284739
Name: count, dtype: int64


In [8]:
X_train.loc[(~(X_train['gender'].duplicated()))]

,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact
id,,,,,,,,,,,,
285927,24.0,7.25,2.95,1.46,2.06,8.14,65.0,155.0,11.54,Male,Medium,Yes
285459,27.0,10.17,3.20,2.42,3.27,8.12,234.0,114.0,11.54,Female,Low,Yes
362545,27.0,2.90,0.78,0.22,1.57,6.60,142.0,39.0,5.67,NaN,NaN,NaN
268214,26.0,10.30,NaN,2.45,1.35,7.74,135.0,35.0,12.68,Other,Medium,Yes


In [9]:
X_train.dtypes

age                        float64
daily_screen_time_hours    float64
social_media_hours         float64
gaming_hours               float64
work_study_hours           float64
sleep_hours                float64
notifications_per_day      float64
app_opens_per_day          float64
weekend_screen_time        float64
gender                         str
stress_level                   str
academic_work_impact           str
dtype: object

C типами данных всё ок - не вижу ничего странного

Заметки по ходу работы с ChatGPT по проекту:
- feature enginnering обычных моделей и моделей градиентного бустинга отличается, но не отменяется целиком. Какие-то признаки всё равно нужно создавать и отбирать: дельта рабочего и выходного дня, неразмеченное время и тд(вот тут поле для размышления не о простых долевых признаках, а о чем-то интереснее - искать бреши и возможности для комбинирования)
- часто пропуски можно оставить незаполненными(модели градиентного бустинга спокойно переваривают их), а кодирование категориальных признаков происходит автоматически(встроено под капотом таких моделей)
- для моделей градиентного бустинга пропуски - тоже информативное значение, поэтому их не стоит по дефолту импутировать
- пропуски в категориальных признаках лучше заменить с NaN на маркер неизвестного значения "unknown" и тп
- здесь полных дубликатов строк дфа нет, но в другом случае, если это не шум, двойное вхождение одной строки тоже имеет вес

Отбор признаков по такой схеме:

- здравого смысла
-       ↓
- baseline CV
-       ↓
- feature importance
-       ↓
- permutation importance / SHAP
-       ↓
- ablation experiments

**Нужно разузнать всё про пункты с 2 по последний**

**Важное уточнение - тюнинг гиперпараметров - поиск гиперпараметров для моделей-кандидатов, чтобы найти лучшую, а fine-tuning - тюнинг лучшей модели для получения лучшей производительности**

- Обе вышеуказанные процедуры можно проводить для CatBoost и XGBoost моделей так и нативными инструментами, так и привычным Grid/RandomizedSearch из sklearn
- *Early stopping* - то, за что мне нужно подшарить во время обучения моделей(это было в 4 главе HOMLP и мне нужно это наконец применить на практике и разобраться с этим вообще)
- О каждом прогоне модели нужно думать, как об эксперименте - нужно соответствующее оформление и использование интерфейсов библиотек для получения всевозможной инфы из тренировок, кросс-валидации и тюнинга моделей

Вот рекомендации от чатагпт по тому, как дальше развивать проект:

1. Сделать CatBoost baseline почти на сырых данных. Убрать очевидный мусор/leakage, оставить числовые NaN, нормально оформить categorical features, без scaling.
2. Сделать правильную validation-схему. Не трогать test до конца. Если классы — начать со StratifiedKFold.
3. Добавить eval_set + early stopping. Начать следить за best_iteration, train/validation metric и историей обучения.
4. Начать feature experiments. Для твоего датасета проверить хотя бы смысловые отношения вроде social_media_share, gaming_share, weekend_screen_delta, notifications_per_app_open. Каждый набор сравнивать через одинаковую CV.
5. Собирать таблицу экспериментов. Например model, features, params, cv_mean, cv_std, best_iteration, train_time. Это очень полезная привычка ещё до MLflow и прочего MLOps.
6. Только после стабильного baseline заниматься hyperparameter tuning. Сначала небольшой RandomizedSearchCV/CatBoost randomized_search, а позже имеет смысл перейти к Optuna.
7. После этого изучить feature importance → permutation importance → SHAP. Это уже следующий уровень интерпретации модели; CatBoost непосредственно предоставляет расчёт feature importance

## Подготовка признаков для Catboost

In [10]:
cat_cols = ['gender', 'stress_level', 'academic_work_impact']

for col in cat_cols:
    X_train[col] = X_train[col].fillna('Missing').astype(str)
    X_val[col] = X_val[col].fillna('Missing').astype(str)
    X_test[col] = X_test[col].fillna('Missing').astype(str)

In [11]:
X_train.dtypes

age                        float64
daily_screen_time_hours    float64
social_media_hours         float64
gaming_hours               float64
work_study_hours           float64
sleep_hours                float64
notifications_per_day      float64
app_opens_per_day          float64
weekend_screen_time        float64
gender                         str
stress_level                   str
academic_work_impact           str
dtype: object

## Создание базового catboost классификатора

In [ ]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=10000
    ,           # Максимальное количество деревьев
    learning_rate=0.02,        # Насколько сильно каждое дерево меняет ансамбль
    depth=None,                # Глубина каждого из деревьев

    loss_function='Logloss',   # Что модель оптимизирует
    eval_metric='AUC',         # Метрика оценки

    random_seed=42,
    verbose=100,

    task_type='GPU'            # Использование видюхи вместо проца для обучения
)

In [13]:
cat_model.fit(
    X_train, 
    y_train,

    cat_features=cat_cols,

    eval_set=(X_val, y_val),

    early_stopping_rounds=200,
    use_best_model=True
)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9064970	best: 0.9064970 (0)	total: 3.37s	remaining: 9h 22m 16s
100:	test: 0.9293338	best: 0.9293338 (100)	total: 5.8s	remaining: 9m 28s
200:	test: 0.9344705	best: 0.9344705 (200)	total: 8.19s	remaining: 6m 39s
300:	test: 0.9379361	best: 0.9379361 (300)	total: 10.5s	remaining: 5m 38s
400:	test: 0.9403647	best: 0.9403647 (400)	total: 12.8s	remaining: 5m 6s
500:	test: 0.9424480	best: 0.9424480 (500)	total: 15.1s	remaining: 4m 45s
600:	test: 0.9441887	best: 0.9441887 (600)	total: 17.3s	remaining: 4m 31s
700:	test: 0.9456831	best: 0.9456831 (700)	total: 19.6s	remaining: 4m 20s
800:	test: 0.9471491	best: 0.9471491 (800)	total: 21.8s	remaining: 4m 10s
900:	test: 0.9482655	best: 0.9482655 (900)	total: 24.2s	remaining: 4m 4s
1000:	test: 0.9493386	best: 0.9493386 (1000)	total: 26.5s	remaining: 3m 58s
1100:	test: 0.9502855	best: 0.9502855 (1100)	total: 28.9s	remaining: 3m 53s
1200:	test: 0.9511092	best: 0.9511092 (1200)	total: 31.2s	remaining: 3m 48s
1300:	test: 0.9517993	best: 0.95179

CatBoostClassifier(eval_metric='AUC', iterations=10000, learning_rate=0.02, loss_function='Logloss', random_seed=42, task_type='GPU', verbose=100)

In [14]:
print(f""" MODEL: baseline CatBoostClassifier

BEST ITERATION:

{cat_model.get_best_iteration()}

BEST SCORE:

{cat_model.get_best_score()}""")

 MODEL: baseline CatBoostClassifier

BEST ITERATION:

9994

BEST SCORE:

{'learn': {'Logloss': 0.21984075573901696}, 'validation': {'Logloss': 0.2297393049759897, 'AUC': 0.9608539938926697}}


## Функция оценивания модели

In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score, 
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
)

def evaluate_classifier(model, X, y, threshold=0.53):
    proba = model.predict_proba(X)[:,1]
    pred = (proba >= threshold).astype(int)

    metrics = pd.Series({
        'roc_auc': roc_auc_score(y,proba),
        'log_loss': log_loss(y,proba),
        'accuracy': accuracy_score(y,pred),
        'precision': precision_score(y,pred, zero_division=0),
        'recall': recall_score(y,pred, zero_division=0),
        'f1': f1_score(y, pred, zero_division=0)
    })

    return metrics, confusion_matrix(y, pred)

In [16]:
cat_metrics, cat_cm = evaluate_classifier(
    cat_model,
    X_val,
    y_val
)

print(cat_metrics)
print(cat_cm)

roc_auc      0.960854
log_loss     0.229740
accuracy     0.898260
precision    0.928124
recall       0.927820
f1           0.927972
dtype: float64
[[16792  3509]
 [ 3525 45311]]


## То же самое на XGBoost

In [17]:
# попросить пояснение от иишки по этой ячейке

for col in cat_cols:
    categories = X_train[col].dropna().unique()

    dtype = pd.CategoricalDtype(
        categories=categories
    )

    X_train[col] = X_train[col].astype(dtype)
    X_val[col] = X_val[col].astype(dtype)
    X_test[col] = X_test[col].astype(dtype)

In [18]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=10000,
    learning_rate=0.02,

    subsample=0.8,
    colsample_bytree=0.8,

    eval_metric='auc',

    enable_categorical=True,

    early_stopping_rounds=200,

    random_state=42,
    n_jobs=-1
)

In [19]:
xgb_model.fit(
    X_train,
    y_train,
    eval_set = [(X_val, y_val)],
    verbose=100,
)

[0]	validation_0-auc:0.88780
[100]	validation_0-auc:0.93525
[200]	validation_0-auc:0.93879
[300]	validation_0-auc:0.94249
[400]	validation_0-auc:0.94685
[500]	validation_0-auc:0.95003
[600]	validation_0-auc:0.95242
[700]	validation_0-auc:0.95430
[800]	validation_0-auc:0.95582
[900]	validation_0-auc:0.95711
[1000]	validation_0-auc:0.95803
[1100]	validation_0-auc:0.95896
[1200]	validation_0-auc:0.95974
[1300]	validation_0-auc:0.96031
[1400]	validation_0-auc:0.96081
[1500]	validation_0-auc:0.96122
[1600]	validation_0-auc:0.96158
[1700]	validation_0-auc:0.96188
[1800]	validation_0-auc:0.96219
[1900]	validation_0-auc:0.96248
[2000]	validation_0-auc:0.96270
[2100]	validation_0-auc:0.96290
[2200]	validation_0-auc:0.96308
[2300]	validation_0-auc:0.96323
[2400]	validation_0-auc:0.96337
[2500]	validation_0-auc:0.96350
[2600]	validation_0-auc:0.96362
[2700]	validation_0-auc:0.96373
[2800]	validation_0-auc:0.96383
[2900]	validation_0-auc:0.96393
[3000]	validation_0-auc:0.96403
[3100]	validation_0-

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",200
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'auc'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [20]:
print(f"""MODEL: XGBoostClassifier

BEST ITERATION

{xgb_model.best_iteration}

BEST SCORE

{xgb_model.best_score}

EVALUATION RESULTS

{xgb_model.evals_result()}
""")

MODEL: XGBoostClassifier

BEST ITERATION

7298

BEST SCORE

0.9650532415922414

EVALUATION RESULTS

{'validation_0': OrderedDict({'auc': [0.8878034951488494, 0.9248845107602852, 0.9261260017044891, 0.9281828658475351, 0.9300498305845538, 0.9294296804708435, 0.9302585081137126, 0.9314533966926594, 0.9318115790274705, 0.9321640992795507, 0.9320325732382407, 0.9319309749882743, 0.9319085662128241, 0.9321641734156655, 0.9320663616551569, 0.9322115156290893, 0.9323935833362897, 0.9322678726932134, 0.9323884896304394, 0.9324850657890339, 0.9325587812949067, 0.9325891922318149, 0.9325008240002218, 0.9324342986888349, 0.9324155679724705, 0.9324999716870647, 0.9325542035158965, 0.9326352539581937, 0.9325397262960807, 0.9324669962457753, 0.9326292141343063, 0.9327502516805104, 0.9328230926828244, 0.9328791839664552, 0.9329534885266283, 0.9329419278316573, 0.9330050751587091, 0.9330503632469914, 0.933050561447625, 0.9330996889797329, 0.9331528289399343, 0.9331569906489122, 0.9331637072800523, 0.9

In [21]:
xgb_metrics, xgb_cm = evaluate_classifier(
    xgb_model,
    X_val,
    y_val
)

print(xgb_metrics, )
xgb_cm

roc_auc      0.965053
log_loss     0.217066
accuracy     0.903293
precision    0.932024
recall       0.930994
f1           0.931509
dtype: float64


array([[16985,  3316],
       [ 3370, 45466]])

In [22]:
comparison = pd.DataFrame({
    'CatBoost': cat_metrics,
    'XGBoost': xgb_metrics
}).T

comparison

,roc_auc,log_loss,accuracy,precision,recall,f1
CatBoost,0.960854,0.229740,0.898260,0.928124,0.927820,0.927972
XGBoost,0.965053,0.217066,0.903293,0.932024,0.930994,0.931509
